# Spotted Python Week 8 Tutorial
In this script, we aim to practice the following skills:
- BPT Diagram
- Fitting a Gaussian
- Determining redshift of a galaxy

In [ ]:
# import libraries
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.io import ascii
from astropy.table import Table, join
from scipy.odr import *
from scipy.signal import find_peaks

## BPT Diagram
### **Example and Exercise:**
Using the classifications given in Kewley et al. (2016), plot the emission line galaxy on a $\log⁡([OIII]/[OII])$ versus $\log⁡([OI]/H_\alpha)$ plot:
1. HII & Composites: $\log⁡([OIII]/[OII])<−1.701 \log⁡([OI]/H_\alpha)−2.163$
2. LINERS: $\log⁡([OIII]/[OII])<1.0 \log⁡([OI]/H_\alpha)+0.7$
3. Seyferts: $\log⁡([OIII]/[OII])>1.0 \log⁡([OI]/H_\alpha)+0.7$


In [ ]:
# Finding the ratios for a chosen object
# SDSS ID: 1237648720693756163
# import data from shared csv file
emission_lines = ascii.read('sdss_spectra.csv', format='basic', delimiter=',')

# define lines
oi = emission_lines['[O I]']
oii = emission_lines['[O II]1'] + emission_lines['[O II]2']
oiii = emission_lines['[O III]']
halpha = emission_lines['[Ha]']

# find ratios
oi_ha = np.log(oi/halpha)
oiii_oii = np.log(oiii/oii)

In [ ]:
emission_lines

In [ ]:
# Plot the diagnostic lines:
# let y = log([OIII]/[OII]) and x = log([OI]/H_alpha)

# find intersection of the diagnostic lines
intersect = (-2.163-0.7)/2.701

# defining the lines
x_hii= np.linspace(-3,1,1000)
y_hii = -1.701*x_hii-2.163
x_liner = np.linspace(intersect,1,1000)
y_liner = x_liner+0.7

# plotting diagnostic lines
plt.plot(x_hii, y_hii, 'black')
plt.plot(x_liner,y_liner, 'black')

# plot the emission galaxy
plt.plot(oi_ha, oiii_oii, '*', color='darkviolet')

# plot display settings
plt.xlabel('$\log$([OI]/H$\\alpha$)')
plt.ylabel('$\log([OIII]/[OII])$')
plt.margins(x=0)
plt.margins(y=0)

# Labels
plt.text(-2.1, -2.5, 'HII', fontsize=12, color = 'crimson', fontweight='bold')
plt.text(-1.3, 1.4, 'Seyferts', fontsize = 12, color = 'mediumblue', fontweight='bold')
plt.text(0, -0.7, 'LINERs', fontsize = 12, color = 'mediumseagreen', fontweight='bold')

## Gaussian Fitting
In pervious weeks, we looked at linear fitting. However gaussian fitting is equally common within physics.

### **Example 1:**
Generate simulated spectral line and fit an gaussian

In [ ]:
# define a general gaussian function
def gaussian(parameters, x):
    c, A, mu, sigma = parameters
    return c + A * np.exp(-(x - mu) ** 2 / (2 * sigma ** 2))

In [ ]:
# generate simulated data
x_sim = np.linspace(1000, 1300, 100) # x values
y_sim = gaussian([10,23,1150,10],x_sim)
y_noise = np.random.normal(10,2,y_sim.shape) # add gaussian noise
y_total = y_sim + y_noise # final y_values

# plot simulated data to check shape
plt.plot(x_sim, y_total, color='royalblue')

In [ ]:
# Fitting a gaussian to the simulated data
model = Model(gaussian) # making the defined function into a model
# takes the real data and fits it to the model with an initial guess for the parameters
odr = ODR(RealData(x_sim, y_total),
                 model, beta0=[100, 20, 1150, 15])
output = odr.run()
output.pprint()

In [ ]:
# plot the simulated data with the fitted gaussian
plt.plot(x_sim, y_total, color = 'royalblue', label='simulated data') # orginal data
plt.plot(x_sim, gaussian(output.beta, x_sim), color='deeppink', label ='fitted gaussian')
plt.legend(loc='best') # show legend

### **Excercise 1:**
To practice this fitting, you can use the below space to simulate your own data.

## Spectroscopic Redshift
Recall that redshift can be determined using the following formulae: $$z=\frac{\Delta \lambda}{\lambda_{rest}}=\frac{\lambda_{observed}-\lambda_{rest}}{\lambda_{rest}}\Rightarrow z=\frac{\lambda_{observed}}{\lambda_{rest}}-1,$$ where $\lambda_{rest}$ is the rest frame wavelength, and $\lambda_{observed}$ is the oberserved wavelength in the spectra.

### **Example 2:**
Open NGC0000.fits, extract flux, plot the spectra, and identify any emission/absorption lines. Attempt to fit the line using a gaussian or alternative methods, and determine the redshift of the example spectra

Notes:
- List of spectral lines available [here](http://astronomy.nmsu.edu/drewski/tableofemissionlines.html#:~:text=The%20ultraviolet%20and%20optical%20spectra,nature%20of%20the%20ionizing%20source.)
- $H_{\alpha}$ (6562.819 A) is almost always the highest in flux, so let's identify that first
- To the left of $H_{\alpha}$ is $[NII]6548$ (6548.050 A)
- To the right of $H_{\alpha}$ is $[NII]6583$ (6583.460 A)

In [ ]:
# Open fits
data = fits.open('NGC0000.fits')
data.info()

In [ ]:
# Let's have a look at the headers
data['data'].header

In [ ]:
# To get the wavelength, we need to use CRPIX3, CRVAL3, CD3_3, and NAXIS3
crpix3 = data['data'].header['crpix3']
crval3 = data['data'].header['crval3']
cdelt3 = data['data'].header['cd3_3']
naxis3 = data['data'].header['naxis3']

wavelength = crval3 + (np.arange(naxis3) + 1 - crpix3) * cdelt3
print(f"wavelength range: [{wavelength[0]} to {wavelength[-1]} Angstroms]")

cube = data['data'].data
print(f"flux cube shape: {cube.shape}")

# Let's just look at the central pixel
flux = cube[:, 15, 15]

# Plot the spectra
plt.figure(figsize=(15,5))
plt.plot(wavelength, flux, color='darkviolet')
plt.xlabel('Wavelength, $\lambda$ $\\AA$') # x-axis label
plt.ylabel('Flux, $F$ $[10^{-20} \\AA cm^{-2} erg s^{-1}]$') # y-axis label

In [ ]:
# let's zoom into the wavelength range of interest
wavelength_short = wavelength[1450:1600]
flux_short = flux[1450:1600]
plt.plot(wavelength_short, flux_short, color='darkviolet')
plt.xlabel('Wavelength, $\lambda$ $\\AA$') # x-axis label
plt.ylabel('Flux, $F$ $[10^{-20} \\AA cm^{-2} erg s^{-1}]$') # y-axis label

In [ ]:
# define the rest frame wavelengths
h_alpha = 6562.819
nii_1 = 6548.050
nii_2 = 6583.460

plt.plot(wavelength_short, flux_short, color='darkviolet')
plt.axvline(h_alpha, color='k')
plt.axvline(nii_1, color='k')
plt.axvline(nii_2, color='k')
plt.xlabel('Wavelength, $\lambda$ $\\AA$') # x-axis label
plt.ylabel('Flux, $F$ $[10^{-20} \\AA cm^{-2} erg s^{-1}]$') # y-axis label

# Identify the wavelengths of the spectral lines
peaks, _ = find_peaks(flux_short, height=4000)
print(f"number of peaks: {len(peaks)}")

# define the observed wavelengths
h_alpha_obs = wavelength_short[peaks[0]] # h_alpha
#nii_1_obs = wavelength_short[peaks[0]] # nii 6548.050
#nii_2_obs = wavelength_short[peaks[2]] # nii 6583.460

# Calculate redshift using individual lines
z_1 = h_alpha_obs/h_alpha -1
#z_2 = nii_1_obs/nii_1 -1
#z_3 = nii_2_obs/nii_2 -1
#z_avg = np.mean((z_1,z_2,z_3))
print(f'Spectroscopic redshift is: z= {round(z_1, 4)}') # prints to 4 decimal places

### **Exercise 2:**
Repeat example 2 with another spectra from the cube, you should get a redshift that is similar (since it is the same galaxy)